In [ ]:
# Setup: Extract src_1 package and install requirements
# Confirm dataset path and package contents
!ls /kaggle/input/
!ls "/kaggle/input/snn-trans-2/"
!ls "/kaggle/input/snn-trans-2/New folder/"

# Copy the uploaded package contents into the Kaggle working directory
!mkdir -p /kaggle/working/src_1
!cp -r "/kaggle/input/snn-trans-2/New folder/"* /kaggle/working/

# Install dependencies from the uploaded requirements file
!pip install -r /kaggle/working/requirements.txt

import sys
sys.path.append('/kaggle/working/src_1')
print("Extracted uploaded package and installed requirements.")

# 🧠 Adaptive Timestep SNN for Brain Tumor Segmentation

**Architecture**: Spiking UNet with Bipolar Linear Self-Attention

**Innovation**: LangGraph-orchestrated agent dynamically assigns fewer timesteps to background pixels, saving compute.

**Training**: Phase 1 (warmup, fixed T) → Phase 2 (agent-guided adaptive T)

In [ ]:
!pip install -q snntorch langgraph nibabel tqdm einops scipy seaborn accelerate bitsandbytes transformers

In [ ]:
import os, sys, glob, json, time, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid')

# ─── CONFIGURATION ───
# Change these paths as needed
KAGGLE_DATA_DIR = "/kaggle/input/datasets/luumsk/asnr-miccai-brats-2023-gli-challenge-training-data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
LOCAL_DATA_DIR = "dataset"
DATA_DIR = KAGGLE_DATA_DIR if os.path.exists(KAGGLE_DATA_DIR) else LOCAL_DATA_DIR

SLICE_SIZE = (128, 128)
BATCH_SIZE = 2
NUM_WORKERS = 2
T = 4           # Max spiking timesteps
TOTAL_EPOCHS = 20
WARMUP_EPOCHS = 10
LR = 1e-3
SEED = 42
SAVE_DIR = "outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

# Seed
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Data dir: {DATA_DIR}")
print(f"Patients found: {len(glob.glob(os.path.join(DATA_DIR, 'BraTS-GLI-*')))}")

## 📦 Dataset — BraTS 2023 (2D Axial Slices)

In [ ]:
class BraTSDataset(Dataset):
    MODALITY_SUFFIXES = ['t1c', 't1n', 't2f', 't2w']

    def __init__(self, data_dir, patient_ids=None, transform=None,
                 slice_size=(128,128), min_brain_fraction=0.02, cache_size=20):
        self.data_dir = data_dir
        self.slice_size = slice_size
        self.transform = transform
        self.min_brain_fraction = min_brain_fraction
        self.cache_size = cache_size
        if patient_ids is None:
            self.patient_dirs = sorted(glob.glob(os.path.join(data_dir, "BraTS-GLI-*")))
        else:
            self.patient_dirs = [os.path.join(data_dir, pid) for pid in patient_ids]
        self.slices = []
        self._cache = {}; self._cache_order = []
        self._build_slice_index()

    def _find_modality_file(self, patient_dir, patient_name, suffix):
        for ext in ['.nii.gz', '.nii']:
            path = os.path.join(patient_dir, f"{patient_name}-{suffix}{ext}")
            if os.path.isfile(path): return path
        dir_path = os.path.join(patient_dir, f"{patient_name}-{suffix}.nii")
        if os.path.isdir(dir_path):
            for f in sorted(os.listdir(dir_path)):
                if f.endswith(('.nii', '.nii.gz')): return os.path.join(dir_path, f)
        raise FileNotFoundError(f"Cannot find {suffix} for {patient_name}")

    def _find_seg_file(self, patient_dir, patient_name):
        for ext in ['.nii.gz', '.nii']:
            path = os.path.join(patient_dir, f"{patient_name}-seg{ext}")
            if os.path.isfile(path): return path
        raise FileNotFoundError(f"Cannot find seg for {patient_name}")

    def _get_slice(self, pdir, patient_name, s_idx):
        img_ch = []
        for suffix in self.MODALITY_SUFFIXES:
            path = self._find_modality_file(pdir, patient_name, suffix)
            slice_data = nib.load(path).dataobj[:, :, s_idx]
            img_ch.append(np.array(slice_data, dtype=np.float32))
        seg_path = self._find_seg_file(pdir, patient_name)
        seg = np.array(nib.load(seg_path).dataobj[:, :, s_idx], dtype=np.int64)
        return np.stack(img_ch, axis=0), seg

    def _build_slice_index(self):
        import json
        idx_path = os.path.join(self.data_dir, "slice_index_cache.json")
        try:
            if os.path.exists(idx_path):
                with open(idx_path, 'r') as f: d = json.load(f)
                if len(d.get('patient_dirs',[])) == len(self.patient_dirs):
                    print(f"Loading cached slice index from {idx_path}...")
                    self.slices = d['slices']; return
        except Exception: pass

        print(f"Building slice index for {len(self.patient_dirs)} patients... (fast method via seg mask)")
        from tqdm import tqdm
        for p_idx, pdir in tqdm(enumerate(self.patient_dirs), total=len(self.patient_dirs)):
            name = os.path.basename(pdir)
            try:
                seg = nib.load(self._find_seg_file(pdir, name)).get_fdata(dtype=np.float32)
                for s in range(seg.shape[-1]):
                    if np.mean(seg[:,:,s] != 0) >= self.min_brain_fraction:
                        self.slices.append((p_idx, s))
            except Exception as e: print(f"  Warning: {name} skipped: {e}")
            
        print(f"  Found {len(self.slices)} valid slices.")
        try:
            p = idx_path if os.access(self.data_dir, os.W_OK) else "slice_index_cache.json"
            with open(p, 'w') as f: json.dump({'patient_dirs':self.patient_dirs, 'slices':self.slices}, f)
        except: pass
        self._cache.clear(); self._cache_order.clear()

    def __len__(self): return len(self.slices)

    def __getitem__(self, idx):
        p_idx, s_idx = self.slices[idx]
        pdir = self.patient_dirs[p_idx]
        img, msk = self._get_slice(pdir, os.path.basename(pdir), s_idx)
        for c in range(4):
            nz = img[c][img[c]!=0]
            if len(nz)>0: img[c] = np.where(img[c]!=0, (img[c]-nz.mean())/(nz.std()+1e-8), 0)
        img = torch.from_numpy(img); msk = torch.from_numpy(msk).long()
        msk[msk==4] = 3
        img = F.interpolate(img.unsqueeze(0), size=self.slice_size, mode='bilinear', align_corners=False).squeeze(0)
        msk = F.interpolate(msk.float().unsqueeze(0).unsqueeze(0), size=self.slice_size, mode='nearest').squeeze(0).squeeze(0).long()
        if self.transform: img, msk = self.transform(img, msk)
        return {'image': img, 'mask': msk}

class BraTSAugmentation:
    def __init__(self, flip_prob=0.5, max_angle=15):
        self.flip_prob = flip_prob; self.max_angle = max_angle
    def __call__(self, image, mask):
        if torch.rand(1).item() < self.flip_prob:
            image = torch.flip(image, [-1]); mask = torch.flip(mask, [-1])
        if torch.rand(1).item() < self.flip_prob*0.5:
            image = torch.flip(image, [-2]); mask = torch.flip(mask, [-2])
        if self.max_angle > 0 and torch.rand(1).item() < 0.5:
            angle = (torch.rand(1).item()*2-1)*self.max_angle
            rad = np.radians(angle)
            theta = torch.tensor([[np.cos(rad),-np.sin(rad),0],[np.sin(rad),np.cos(rad),0]], dtype=torch.float32).unsqueeze(0)
            grid = F.affine_grid(theta, image.unsqueeze(0).size(), align_corners=False)
            image = F.grid_sample(image.unsqueeze(0), grid, mode='bilinear', align_corners=False).squeeze(0)
            mask = F.grid_sample(mask.float().unsqueeze(0).unsqueeze(0), grid, mode='nearest', align_corners=False).squeeze(0).squeeze(0).long()
        if torch.rand(1).item() < 0.3: image = image * (0.9 + torch.rand(1).item()*0.2)
        return image, mask

def get_dataloaders(data_dir, batch_size=2, val_split=0.2, slice_size=(128,128), num_workers=2, seed=42):
    patient_dirs = sorted(glob.glob(os.path.join(data_dir, "BraTS-GLI-*")))
    patient_ids = [os.path.basename(d) for d in patient_dirs]
    np.random.seed(seed); np.random.shuffle(patient_ids)
    n_val = max(1, int(len(patient_ids) * val_split))
    val_ids, train_ids = patient_ids[:n_val], patient_ids[n_val:]
    print(f"Train: {len(train_ids)} patients, Val: {len(val_ids)} patients")
    train_ds = BraTSDataset(data_dir, train_ids, BraTSAugmentation(), slice_size)
    val_ds = BraTSDataset(data_dir, val_ids, None, slice_size)
    return {
        'train': DataLoader(train_ds, batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True),
        'val': DataLoader(val_ds, batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    }

# Quick test
dls = get_dataloaders(DATA_DIR, BATCH_SIZE, slice_size=SLICE_SIZE)
batch = next(iter(dls['train']))
print(f"Image: {batch['image'].shape}, Mask: {batch['mask'].shape}")
print(f"Mask classes: {torch.unique(batch['mask']).tolist()}")

### 🔍 Sample Visualization

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i in range(2):
    b = next(iter(dls['train']))
    img, msk = b['image'][0], b['mask'][0]
    for j, (title, data) in enumerate([
        ('T1CE', img[0]), ('T1N', img[1]), ('T2-FLAIR', img[2]), ('T2W', img[3]), ('Mask', msk)
    ]):
        ax = axes[i, j]
        cmap = 'Set1' if j == 4 else 'gray'
        ax.imshow(data.numpy(), cmap=cmap)
        if i == 0: ax.set_title(title, fontsize=13, fontweight='bold')
        ax.axis('off')
plt.suptitle('BraTS Sample Slices', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/samples.png', dpi=150, bbox_inches='tight'); plt.show()

## 🧬 Spiking UNet with Bipolar Linear Self-Attention

In [ ]:
import snntorch as snn
from snntorch import surrogate

class SpikingConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, beta=0.5):
        super().__init__()
        self.conv1=nn.Conv2d(in_ch,out_ch,3,padding=1,bias=False); self.bn1=nn.BatchNorm2d(out_ch)
        self.lif1=snn.Leaky(beta=beta,spike_grad=surrogate.fast_sigmoid(),init_hidden=False)
        self.conv2=nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False); self.bn2=nn.BatchNorm2d(out_ch)
        self.lif2=snn.Leaky(beta=beta,spike_grad=surrogate.fast_sigmoid(),init_hidden=False)
    def init_mem(self, B, C, H, W, dev):
        return torch.zeros(B,C,H,W,device=dev), torch.zeros(B,C,H,W,device=dev)
    def forward(self, x, mems=None):
        B,_,H,W=x.shape; C=self.conv1.out_channels
        if mems is None: m1,m2=self.init_mem(B,C,H,W,x.device)
        else: m1,m2=mems
        s1,m1=self.lif1(self.bn1(self.conv1(x)),m1)
        s2,m2=self.lif2(self.bn2(self.conv2(s1)),m2)
        return s2,(m1,m2)

class BipolarLinearAttention(nn.Module):
    def __init__(self, dim, num_heads=4, beta=0.5):
        super().__init__()
        self.num_heads=num_heads; self.head_dim=dim//num_heads
        self.q_proj=nn.Conv2d(dim,dim,1,bias=False); self.k_proj=nn.Conv2d(dim,dim,1,bias=False)
        self.v_proj=nn.Conv2d(dim,dim,1,bias=False); self.out_proj=nn.Conv2d(dim,dim,1,bias=False)
        self.q_lif=snn.Leaky(beta=beta,spike_grad=surrogate.fast_sigmoid(),init_hidden=False)
        self.k_lif=snn.Leaky(beta=beta,spike_grad=surrogate.fast_sigmoid(),init_hidden=False)
        self.v_lif=snn.Leaky(beta=beta,spike_grad=surrogate.fast_sigmoid(),init_hidden=False)
        self.bn=nn.BatchNorm2d(dim); self.scale=self.head_dim**-0.5
    def init_mem(self, B, C, H, W, dev):
        z=torch.zeros(B,C,H,W,device=dev); return z.clone(),z.clone(),z.clone()
    def forward(self, x, mems=None):
        B,C,H,W=x.shape
        if mems is None: qm,km,vm=self.init_mem(B,C,H,W,x.device)
        else: qm,km,vm=mems
        qs,qm=self.q_lif(self.q_proj(x),qm); ks,km=self.k_lif(self.k_proj(x),km); vs,vm=self.v_lif(self.v_proj(x),vm)
        q=2.*qs-1.; k=2.*ks-1.; v=2.*vs-1.
        q=q.view(B,self.num_heads,self.head_dim,H*W).permute(0,1,3,2)
        k=k.view(B,self.num_heads,self.head_dim,H*W).permute(0,1,3,2)
        v=v.view(B,self.num_heads,self.head_dim,H*W).permute(0,1,3,2)
        q=F.elu(q)+1.; k=F.elu(k)+1.
        kv=torch.matmul(k.transpose(-2,-1),v)  # (B,h,d,d) — linear!
        out=torch.matmul(q,kv)*self.scale
        denom=torch.matmul(q,k.sum(dim=-2,keepdim=True).transpose(-2,-1))+1e-6
        out=out/denom
        out=out.permute(0,1,3,2).contiguous().view(B,C,H,W)
        return self.bn(self.out_proj(out)),(qm,km,vm)

class SpikingUNet(nn.Module):
    def __init__(self, in_ch=4, n_cls=4, base=32, beta=0.5):
        super().__init__()
        self.enc1=SpikingConvBlock(in_ch,base,beta); self.enc2=SpikingConvBlock(base,base*2,beta)
        self.enc3=SpikingConvBlock(base*2,base*4,beta); self.enc4=SpikingConvBlock(base*4,base*8,beta)
        self.pool=nn.MaxPool2d(2)
        self.attn=BipolarLinearAttention(base*8,4,beta); self.attn_conv=SpikingConvBlock(base*8,base*8,beta)
        self.up4=nn.ConvTranspose2d(base*8,base*4,2,stride=2); self.dec4=SpikingConvBlock(base*12,base*4,beta)
        self.up3=nn.ConvTranspose2d(base*4,base*2,2,stride=2); self.dec3=SpikingConvBlock(base*6,base*2,beta)
        self.up2=nn.ConvTranspose2d(base*2,base,2,stride=2); self.dec2=SpikingConvBlock(base*3,base,beta)
        self.up1=nn.ConvTranspose2d(base,base//2,2,stride=2); self.dec1=SpikingConvBlock(base//2+base,base//2,beta)
        self.head=nn.Conv2d(base//2,n_cls,1)
    def _shapes(self,H,W):
        s=[(H,W)];
        for _ in range(4): H,W=H//2,W//2; s.append((H,W))
        return s
    def init_all_mems(self,B,H,W,dev):
        ch=self.enc1.conv1.out_channels; s=self._shapes(H,W); m={}
        m['e1']=self.enc1.init_mem(B,ch,*s[0],dev); m['e2']=self.enc2.init_mem(B,ch*2,*s[1],dev)
        m['e3']=self.enc3.init_mem(B,ch*4,*s[2],dev); m['e4']=self.enc4.init_mem(B,ch*8,*s[3],dev)
        m['at']=self.attn.init_mem(B,ch*8,*s[4],dev); m['ac']=self.attn_conv.init_mem(B,ch*8,*s[4],dev)
        m['d4']=self.dec4.init_mem(B,ch*4,*s[3],dev); m['d3']=self.dec3.init_mem(B,ch*2,*s[2],dev)
        m['d2']=self.dec2.init_mem(B,ch,*s[1],dev); m['d1']=self.dec1.init_mem(B,ch//2,*s[0],dev)
        return m
    @staticmethod
    def _pad_cat(up,skip):
        dH=skip.size(2)-up.size(2); dW=skip.size(3)-up.size(3)
        up=F.pad(up,[dW//2,dW-dW//2,dH//2,dH-dH//2])
        return torch.cat([up,skip],dim=1)
    def forward_one_timestep(self,x,m):
        e1,m['e1']=self.enc1(x,m['e1']); e2,m['e2']=self.enc2(self.pool(e1),m['e2'])
        e3,m['e3']=self.enc3(self.pool(e2),m['e3']); e4,m['e4']=self.enc4(self.pool(e3),m['e4'])
        b=self.pool(e4); b,m['at']=self.attn(b,m['at']); b,m['ac']=self.attn_conv(b,m['ac'])
        d4=self._pad_cat(self.up4(b),e4); d4,m['d4']=self.dec4(d4,m['d4'])
        d3=self._pad_cat(self.up3(d4),e3); d3,m['d3']=self.dec3(d3,m['d3'])
        d2=self._pad_cat(self.up2(d3),e2); d2,m['d2']=self.dec2(d2,m['d2'])
        d1=self._pad_cat(self.up1(d2),e1); d1,m['d1']=self.dec1(d1,m['d1'])
        return self.head(d1),m

class AdaptiveTimestepSNN(nn.Module):
    def __init__(self, in_ch=4, n_cls=4, T=4, base=32, beta=0.5):
        super().__init__()
        self.T=T; self.n_cls=n_cls; self.unet=SpikingUNet(in_ch,n_cls,base,beta)
    def forward(self, x, timestep_map=None):
        B,C,H,W=x.shape; dev=x.device
        m=self.unet.init_all_mems(B,H,W,dev)
        s=torch.zeros(B,self.n_cls,H,W,device=dev)
        if timestep_map is None:
            for t in range(self.T): l,m=self.unet.forward_one_timestep(x,m); s+=l
            return s/self.T
        else:
            cnt=timestep_map.unsqueeze(1).float()
            for t in range(self.T):
                l,m=self.unet.forward_one_timestep(x,m)
                active=(timestep_map>t).unsqueeze(1).float()
                s+=l*active
            return s/cnt.clamp(min=1.)
    def forward_single_timestep(self, x, mems=None):
        B,C,H,W=x.shape
        if mems is None: mems=self.unet.init_all_mems(B,H,W,x.device)
        l,mems=self.unet.forward_one_timestep(x,mems)
        return l,mems

# Test
model = AdaptiveTimestepSNN(4, 4, T, 32, 0.5).to(device)
x = torch.randn(1, 4, 128, 128).to(device)
with torch.no_grad(): out = model(x)
print(f"Model output: {out.shape}")
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,} ({n_params*4/1024/1024:.1f} MB)")

## 🤖 Adaptive Timestep Agent (CNN + LangGraph)

In [ ]:
class CNNTimestepAgent(nn.Module):
    def __init__(self, in_ch=4, n_cls=4, T=4, patch_size=8):
        super().__init__()
        self.T=T; self.patch_size=patch_size
        self.net=nn.Sequential(
            nn.Conv2d(in_ch+n_cls,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32,1,1))
    def forward(self, image, init_logits, threshold=0.3):
        x=torch.cat([image, init_logits.detach()], dim=1)
        conf=torch.sigmoid(self.net(x))
        if self.patch_size>1:
            cp=F.avg_pool2d(conf,self.patch_size,self.patch_size)
            cp=F.interpolate(cp,size=conf.shape[2:],mode='nearest')
        else: cp=conf
        soft_t=1.+(self.T-1.)*(1.-cp.squeeze(1))
        t_map=soft_t.round().clamp(1,self.T).int()
        return t_map, conf, soft_t

def compute_gradient_magnitude(image):
    gray=image.mean(dim=1,keepdim=True)
    sx=torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]],dtype=image.dtype,device=image.device).view(1,1,3,3)
    sy=torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]],dtype=image.dtype,device=image.device).view(1,1,3,3)
    gx=F.conv2d(gray,sx,padding=1); gy=F.conv2d(gray,sy,padding=1)
    g=torch.sqrt(gx**2+gy**2+1e-8)
    for b in range(g.size(0)):
        mx=g[b].max()
        if mx>0: g[b]=g[b]/mx
    return g.squeeze(1)

def compute_entropy(logits):
    p=F.softmax(logits,dim=1); lp=torch.log(p+1e-8)
    e=-(p*lp).sum(dim=1)
    return e/np.log(logits.size(1))

def compute_hybrid_uncertainty(logits, image, alpha=0.6, beta=0.4):
    return alpha*compute_entropy(logits)+beta*compute_gradient_magnitude(image)

# LangGraph pipeline (optional)
try:
    from langgraph.graph import StateGraph, END
    HAS_LANGGRAPH = True
    print("LangGraph available ✓")
except ImportError:
    HAS_LANGGRAPH = False
    print("LangGraph not found, using direct pipeline")

agent = CNNTimestepAgent(4, 4, T).to(device)
with torch.no_grad():
    init_l, _ = model.forward_single_timestep(x)
    t_map, conf, _ = agent(x, init_l)
print(f"Timestep map: {t_map.shape}, unique values: {torch.unique(t_map).tolist()}")
print(f"Confidence: {conf.shape}, range: [{conf.min():.3f}, {conf.max():.3f}]")

## 📊 Metrics — Dice, Hausdorff, Energy Tracking

In [ ]:
from scipy.spatial.distance import directed_hausdorff
from scipy.spatial import cKDTree

def dice_score(pred, target, n_cls=4, smooth=1e-5):
    per_class={}; vals=[]
    for c in range(n_cls):
        pc=(pred==c).float(); tc=(target==c).float()
        inter=(pc*tc).sum(); union=pc.sum()+tc.sum()
        d=(2.*inter+smooth)/(union+smooth)
        per_class[c]=d.item()
        if c>0: vals.append(d.item())
    return per_class, np.mean(vals) if vals else 0.

class DiceLoss(nn.Module):
    def __init__(self, n_cls=4, smooth=1e-5):
        super().__init__(); self.n_cls=n_cls; self.smooth=smooth
    def forward(self, logits, targets):
        probs=torch.softmax(logits,dim=1)
        toh=F.one_hot(targets,self.n_cls).permute(0,3,1,2).float()
        loss=0.
        for c in range(1,self.n_cls):
            p=probs[:,c]; t=toh[:,c]
            inter=(p*t).sum(dim=(-1,-2)); union=p.sum(dim=(-1,-2))+t.sum(dim=(-1,-2))
            loss+=1.-(2.*inter+self.smooth)/(union+self.smooth)
        return (loss/(self.n_cls-1)).mean()

def hausdorff_distance_95(pred, target, n_cls=4):
    if torch.is_tensor(pred): pred=pred.cpu().numpy()
    if torch.is_tensor(target): target=target.cpu().numpy()
    hds=[]
    for c in range(1,n_cls):
        pp=np.argwhere(pred==c); tp=np.argwhere(target==c)
        if len(pp)==0 and len(tp)==0: hds.append(0.); continue
        if len(pp)==0 or len(tp)==0: continue
        d1,_=cKDTree(tp).query(pp); d2,_=cKDTree(pp).query(tp)
        hds.append(np.percentile(np.concatenate([d1,d2]),95))
    return np.mean(hds) if hds else 0.

class EnergyTracker:
    def __init__(self, T_max, img_size=(128,128)):
        self.T_max=T_max; self.H,self.W=img_size; self.total_px=self.H*self.W
        self.baseline=0; self.actual=0; self.n_img=0
    def update(self, t_map):
        if torch.is_tensor(t_map): t_map=t_map.cpu().numpy()
        for i in range(t_map.shape[0]):
            self.baseline+=self.total_px*self.T_max
            self.actual+=t_map[i].sum(); self.n_img+=1
    def get_savings(self): return 1.-(self.actual/self.baseline) if self.baseline>0 else 0.
    def summary(self):
        return {'n_images':self.n_img,'baseline_ops':self.baseline,'actual_ops':self.actual,
                'savings_pct':self.get_savings()*100,'avg_t':self.actual/max(1,self.n_img*self.total_px)}

print("Metrics ready ✓")

## 🚀 Training — Phase 1 (Warmup) + Phase 2 (Agent)

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator(mixed_precision='fp16')

# Re-create model and agent for accelerate
model = AdaptiveTimestepSNN(4, 4, T, 32, 0.5)
agent = CNNTimestepAgent(4, 4, T)
optimizer = torch.optim.AdamW(list(model.parameters())+list(agent.parameters()), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)
dice_loss_fn = DiceLoss(4)
ce_loss_fn = nn.CrossEntropyLoss()

dls = get_dataloaders(DATA_DIR, BATCH_SIZE, slice_size=SLICE_SIZE, num_workers=NUM_WORKERS)

model, agent, optimizer, scheduler = accelerator.prepare(model, agent, optimizer, scheduler)
train_dl = accelerator.prepare(dls['train']); val_dl = accelerator.prepare(dls['val'])
dice_loss_fn = dice_loss_fn.to(accelerator.device)

history = {'train_loss':[],'val_loss':[],'train_dice':[],'val_dice':[],'val_hd95':[],'energy_savings':[]}
best_dice = 0.0; start = time.time()

for epoch in range(1, TOTAL_EPOCHS+1):
    use_agent = epoch > WARMUP_EPOCHS
    phase = "Phase 2 (Agent)" if use_agent else "Phase 1 (Warmup)"
    if accelerator.is_main_process:
        print(f"\n{'─'*50}\nEpoch {epoch}/{TOTAL_EPOCHS} — {phase} | LR: {scheduler.get_last_lr()[0]:.2e}")

    # ── Train ──
    model.train(); agent.train()
    t_loss_sum=0.; t_dice_sum=0.; t_count=0
    energy_tr = EnergyTracker(T, SLICE_SIZE)
    for batch in tqdm(train_dl, desc="Train", disable=not accelerator.is_main_process):
        imgs, msks = batch['image'], batch['mask']
        optimizer.zero_grad()
        if use_agent:
            with torch.no_grad(): il,_=model.forward_single_timestep(imgs)
            tm,_,_=agent(imgs,il); logits=model(imgs,timestep_map=tm); energy_tr.update(tm)
        else:
            logits=model(imgs)
        loss=dice_loss_fn(logits,msks)+ce_loss_fn(logits,msks)
        accelerator.backward(loss); accelerator.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        preds=logits.argmax(dim=1); _,md_=dice_score(preds,msks,4)
        t_loss_sum+=loss.item(); t_dice_sum+=md_; t_count+=1

    # ── Val ──
    model.eval(); agent.eval()
    v_loss_sum=0.; v_dice_sum=0.; v_count=0; v_hd95s=[]
    energy_val = EnergyTracker(T, SLICE_SIZE)
    sample_data = {'images':[],'masks':[],'preds':[]}
    with torch.no_grad():
        for batch in tqdm(val_dl, desc="Val  ", disable=not accelerator.is_main_process):
            imgs, msks = batch['image'], batch['mask']
            if use_agent:
                il,_=model.forward_single_timestep(imgs)
                tm,_,_=agent(imgs,il); logits=model(imgs,timestep_map=tm); energy_val.update(tm)
            else:
                logits=model(imgs)
            loss=dice_loss_fn(logits,msks)+ce_loss_fn(logits,msks)
            preds=logits.argmax(dim=1); _,md_=dice_score(preds,msks,4)
            v_loss_sum+=loss.item(); v_dice_sum+=md_; v_count+=1
            try:
                hd=hausdorff_distance_95(preds,msks,4)
                if hd<float('inf'): v_hd95s.append(hd)
            except: pass
            if len(sample_data['images'])<4:
                sample_data['images'].append(imgs[:1].cpu())
                sample_data['masks'].append(msks[:1].cpu())
                sample_data['preds'].append(preds[:1].cpu())

    scheduler.step()

    tl=t_loss_sum/max(1,t_count); td=t_dice_sum/max(1,t_count)
    vl=v_loss_sum/max(1,v_count); vd=v_dice_sum/max(1,v_count)
    vh=np.mean(v_hd95s) if v_hd95s else 0.
    sv=energy_val.get_savings()*100 if use_agent else 0.

    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_dice'].append(td); history['val_dice'].append(vd)
    history['val_hd95'].append(vh); history['energy_savings'].append(sv)

    if accelerator.is_main_process:
        elapsed=(time.time()-start)/60
        print(f"  Train — loss:{tl:.4f} dice:{td:.4f}")
        print(f"  Val   — loss:{vl:.4f} dice:{vd:.4f} HD95:{vh:.2f}")
        if use_agent: print(f"  Energy saved: {sv:.1f}%")
        print(f"  Time: {elapsed:.1f}min")
        if vd>best_dice:
            best_dice=vd
            torch.save(accelerator.unwrap_model(model).state_dict(), f'{SAVE_DIR}/best_model.pt')
            print(f"  ★ Best val dice: {best_dice:.4f}")

total_time=(time.time()-start)/60
print(f"\n{'='*50}\nDone in {total_time:.1f} min | Best dice: {best_dice:.4f}\n{'='*50}")
with open(f'{SAVE_DIR}/history.json','w') as f: json.dump(history,f,indent=2)

## 📈 Results & Visualization

In [ ]:
# Save checkpoint for resuming tomorrow
checkpoint_path = f'{SAVE_DIR}/checkpoint_epoch8.pt'
checkpoint = {
    'epoch': 8,  # Update this to current epoch
    'model_state': accelerator.unwrap_model(model).state_dict(),
    'agent_state': accelerator.unwrap_model(agent).state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'scheduler_state': scheduler.state_dict(),
    'history': history,
    'best_dice': best_dice,
    'config': {
        'T': T,
        'LR': LR,
        'BATCH_SIZE': BATCH_SIZE,
        'TOTAL_EPOCHS': TOTAL_EPOCHS,
        'WARMUP_EPOCHS': WARMUP_EPOCHS
    }
}
torch.save(checkpoint, checkpoint_path)
print(f"✓ Checkpoint saved to {checkpoint_path}")
print(f"✓ File size: {os.path.getsize(checkpoint_path) / 1024 / 1024:.1f} MB")

# Display checkpoint info
print(f"\n📋 Checkpoint Info:")
print(f"  Epoch: {checkpoint['epoch']}/20")
print(f"  Best Val Dice: {checkpoint['best_dice']:.4f}")
print(f"  Total Training Time: {total_time:.1f} min")
print(f"  Next phase: Phase 2 (Agent enabled) starts at epoch 11")
print(f"\n💾 Download '{checkpoint_path}' to resume training tomorrow.")

In [ ]:
# ═══ RESUME FROM CHECKPOINT (Run this at start of next session) ═══
# Upload the checkpoint file you downloaded, then run this cell to load it

checkpoint_path_resume = f'{SAVE_DIR}/checkpoint_epoch8.pt'

if os.path.exists(checkpoint_path_resume):
    print(f"Loading checkpoint from {checkpoint_path_resume}...")
    checkpoint = torch.load(checkpoint_path_resume, map_location=device)
    
    # Reload model and agent
    model.load_state_dict(checkpoint['model_state'])
    agent.load_state_dict(checkpoint['agent_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    scheduler.load_state_dict(checkpoint['scheduler_state'])
    
    # Restore history and best dice
    history = checkpoint['history']
    best_dice = checkpoint['best_dice']
    resume_epoch = checkpoint['epoch'] + 1
    
    print(f"✓ Checkpoint loaded successfully!")
    print(f"  Resuming from epoch {resume_epoch}/20")
    print(f"  Best Val Dice: {best_dice:.4f}")
    print(f"  History entries: {len(history['train_loss'])}")
else:
    print(f"Checkpoint not found at {checkpoint_path_resume}")
    print("Starting fresh training from epoch 1...")

In [ ]:
# Loss curves
fig,ax=plt.subplots(1,1,figsize=(10,6))
ax.plot(history['train_loss'],'b-o',label='Train Loss',ms=4)
ax.plot(history['val_loss'],'r-o',label='Val Loss',ms=4)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Loss Curves',fontweight='bold')
ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/loss_curves.png',dpi=150); plt.show()

# Dice curves
fig,ax=plt.subplots(1,1,figsize=(10,6))
ax.plot(history['train_dice'],'b-o',label='Train Dice',ms=4)
ax.plot(history['val_dice'],'g-o',label='Val Dice',ms=4)
ax.set_xlabel('Epoch'); ax.set_ylabel('Dice'); ax.set_title('Dice Score',fontweight='bold')
ax.legend(); ax.grid(True,alpha=0.3); ax.set_ylim(0,1)
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/dice_curves.png',dpi=150); plt.show()

# HD95
fig,ax=plt.subplots(1,1,figsize=(10,6))
ax.plot(history['val_hd95'],'m-o',label='Val HD95',ms=4)
ax.set_xlabel('Epoch'); ax.set_ylabel('HD95 (px)'); ax.set_title('Hausdorff Distance 95',fontweight='bold')
ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/hd95.png',dpi=150); plt.show()

# Sample predictions
if sample_data['images']:
    imgs_=torch.cat(sample_data['images']); msks_=torch.cat(sample_data['masks']); preds_=torch.cat(sample_data['preds'])
    n=min(4,len(imgs_))
    fig,axes=plt.subplots(n,4,figsize=(16,4*n))
    if n==1: axes=axes[np.newaxis,:]
    cmap=plt.cm.get_cmap('Set1',4)
    for i in range(n):
        axes[i,0].imshow(imgs_[i,0].numpy(),cmap='gray'); axes[i,0].axis('off')
        axes[i,1].imshow(imgs_[i].numpy().mean(0),cmap='gray'); axes[i,1].axis('off')
        axes[i,2].imshow(msks_[i].numpy(),cmap=cmap,vmin=0,vmax=3); axes[i,2].axis('off')
        axes[i,3].imshow(preds_[i].numpy(),cmap=cmap,vmin=0,vmax=3); axes[i,3].axis('off')
        if i==0:
            axes[i,0].set_title('T1CE'); axes[i,1].set_title('Avg Modalities')
            axes[i,2].set_title('Ground Truth'); axes[i,3].set_title('Prediction')
    plt.suptitle('Sample Predictions',fontsize=14,fontweight='bold',y=1.01)
    plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/sample_preds.png',dpi=150,bbox_inches='tight'); plt.show()

## ⚡ Energy Savings Analysis

In [ ]:
# Energy savings over epochs
fig,axes=plt.subplots(1,2,figsize=(14,5))
ax=axes[0]
ax.plot(range(1,TOTAL_EPOCHS+1), history['energy_savings'],'g-o',ms=4)
ax.axvline(x=WARMUP_EPOCHS+0.5, color='red', linestyle='--', label='Agent enabled')
ax.set_xlabel('Epoch'); ax.set_ylabel('Energy Saved (%)'); ax.set_title('Energy Savings per Epoch',fontweight='bold')
ax.legend(); ax.grid(True,alpha=0.3)

ax=axes[1]
savings=history['energy_savings']
phase2_savings = [s for s in savings if s > 0]
if phase2_savings:
    avg_savings = np.mean(phase2_savings)
    ax.bar(['Baseline\n(Full T)','Adaptive\n(Agent)'],[100, 100-avg_savings],
           color=['#e74c3c','#2ecc71'],width=0.5,edgecolor='black')
    ax.set_ylabel('Relative Compute (%)'); ax.set_title(f'Avg Savings: {avg_savings:.1f}%',fontweight='bold')
    ax.grid(True,alpha=0.3,axis='y')
else:
    ax.text(0.5,0.5,'No agent phase data',ha='center',va='center',transform=ax.transAxes)

plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/energy.png',dpi=150); plt.show()

# Final summary
print(f"""
{'='*50}
FINAL RESULTS
{'='*50}
Best Val Dice:     {best_dice:.4f}
Final Val HD95:    {history['val_hd95'][-1]:.2f}
Avg Energy Saved:  {np.mean(phase2_savings) if phase2_savings else 0:.1f}%
Training Time:     {total_time:.1f} min
{'='*50}
""")